### Structured Outputs

In [1]:
!pip install openai -q

In [2]:
import os
import requests
from openai import OpenAI
from typing import List, Optional
from pydantic import BaseModel, Field

In [3]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [4]:
# Definimos la API Key para vincular el notebook con nuestra cuenta de OpenAI
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

In [5]:
client = OpenAI()

In [6]:
alphavantage_api = userdata.get('ALPHAVANTAGE_API')
ticker = 'NVDA'
quarter = '2026Q2'

In [7]:
url_transcripts = f'https://www.alphavantage.co/query?function=EARNINGS_CALL_TRANSCRIPT&symbol={ticker}&quarter={quarter}&apikey={alphavantage_api}'
response_transcripts = requests.get(url_transcripts)
call_transcripts = response_transcripts.json()

In [8]:
type(call_transcripts)

dict

In [9]:
call_transcripts

{'symbol': 'NVDA',
 'quarter': '2026Q2',
 'transcript': [{'speaker': 'Operator',
   'title': 'Operator',
   'content': "Good afternoon. My name is Sarah, and I will be your conference operator today. At this time, I would like to welcome everyone to NVIDIA Corporation's Second Quarter Fiscal 2026 Financial Results Conference Call. All lines have been placed on mute to prevent any background noise. After the speakers' remarks, there will be a question and answer session. If you would like to ask a question during this time, simply press star followed by the number one on your telephone keypad. If you would like to withdraw your question, press star 1 again. Thank you. Toshiya Hari, you may begin your conference. Thank you.",
   'sentiment': '0.0'},
  {'speaker': 'Operator',
   'title': 'Operator',
   'content': "Good afternoon, everyone, and welcome to NVIDIA Corporation's conference call for 2026. With me today from NVIDIA Corporation are Jensen Huang, president and chief executive off

In [10]:
call_transcripts.keys()

dict_keys(['symbol', 'quarter', 'transcript'])

In [11]:
type(call_transcripts['transcript'])

list

In [12]:
len(call_transcripts['transcript'])

29

In [13]:
call_transcripts['transcript'][0]

{'speaker': 'Operator',
 'title': 'Operator',
 'content': "Good afternoon. My name is Sarah, and I will be your conference operator today. At this time, I would like to welcome everyone to NVIDIA Corporation's Second Quarter Fiscal 2026 Financial Results Conference Call. All lines have been placed on mute to prevent any background noise. After the speakers' remarks, there will be a question and answer session. If you would like to ask a question during this time, simply press star followed by the number one on your telephone keypad. If you would like to withdraw your question, press star 1 again. Thank you. Toshiya Hari, you may begin your conference. Thank you.",
 'sentiment': '0.0'}

In [15]:
call_transcripts['transcript'][2]

{'speaker': 'Colette Kress',
 'title': 'CFO',
 'content': "We delivered another record quarter while navigating what continues to be a dynamic external environment. Total revenue was $46.7 billion, exceeding our outlook as we grew sequentially across all market platforms. Data center revenue grew 56% year over year. Data center revenue also grew sequentially despite the $4 billion decline in H20 revenue. NVIDIA Corporation's Blackwell platform reached record levels, growing sequentially by 17%. We began production shipments of GB300 in Q2. Our full stack AI solutions for cloud service providers, Neo Clouds, enterprises, and sovereigns are all contributing to our growth. We are at the beginning of an industrial revolution that will transform every industry. We see $3 to $4 trillion in AI infrastructure spend by the end of the decade. The scale and scope of these build-outs present significant long-term growth opportunities for NVIDIA Corporation. The GB200 NBL system is seeing widesprea

In [16]:
call_transcripts['transcript'][-1]

{'speaker': 'Operator',
 'title': 'Operator',
 'content': "This concludes today's conference call. You may now disconnect.",
 'sentiment': '0.0'}

In [17]:
class EarningsCallInsights(BaseModel):
    """
    Output estructurado con insights clave en español extraídos de una earnings call.
    """
    symbol: Optional[str] = Field(description="Ticker de la compañía, ej. AAPL, AMZN, MSFT, etc")
    sentiment: Optional[str] = Field(description="Sentimiento general: Muy bajista / Bajista / Alcista / Muy alcista")
    summary: str = Field(description="Resumen ejecutivo de la llamada en español (párrafo de 3 líneas)")
    key_topics: List[str] = Field(default_factory=list, description="Temas estratégicos principales, cado uno con un máximo de 4 palabras")
    guidance: List[str] = Field(default_factory=list, description="Guías o proyecciones futuras mencionadas, en español")
    numeric_highlights: List[str] = Field(default_factory=list, description="Cifras o métricas clave reportadas, en español")
    risks: List[str] = Field(default_factory=list, description="Riesgos explícitos o implícitos discutidos, en español")
    catalysts: List[str] = Field(default_factory=list, description="Catalizadores futuros o eventos relevantes, en español")
    analyst_questions: List[str] = Field(default_factory=list, description="Top 3 preguntas destacadas de analistas, en español")
    unanswered_topics: List[str] = Field(default_factory=list, description="Temas abiertos o sin respuesta clara, en español")
    bullish_points: List[str] = Field(default_factory=list, description="Tesis alcistas derivadas de la llamada, en español")
    bearish_points: List[str] = Field(default_factory=list, description="Tesis bajistas derivadas de la llamada, en español")
    red_flags: List[str] = Field(default_factory=list, description="Alertas o señales negativas detectadas, en español")
    emotion: Optional[str] = Field(description="Emoción general: Optimismo / Incertidumbre / Preocupación / Entusiasmo / Frustración")

In [18]:
def render_transcript(d: dict) -> str:
    """
    Renders the transcript data from a dictionary into a formatted string.

    Args:
        d: A dictionary containing the transcript data. Expected keys are 'symbol',
           'quarter', and 'transcript'. The 'transcript' key should contain a list
           of dictionaries, each with 'speaker', 'title' (optional), and 'content'.

    Returns:
        A formatted string representation of the transcript.
    """
    header_info = f"Symbol: {d.get('symbol','')} | Quarter: {d.get('quarter','')}"
    transcript_lines = [header_info, "TRANSCRIPT:"]
    for index, transcript_entry in enumerate(d.get("transcript", [])):
        speaker_name = transcript_entry.get("speaker", "Unknown")
        speaker_title = transcript_entry.get("title")
        utterance_text = (transcript_entry.get("content") or "").strip()
        speaker_tag = f"{speaker_name} ({speaker_title})" if speaker_title else speaker_name
        transcript_lines.append(f"[{index:04d}] {speaker_tag}: {utterance_text}")
    return "\n".join(transcript_lines)

In [19]:
render_transcript(call_transcripts)

"Symbol: NVDA | Quarter: 2026Q2\nTRANSCRIPT:\n[0000] Operator (Operator): Good afternoon. My name is Sarah, and I will be your conference operator today. At this time, I would like to welcome everyone to NVIDIA Corporation's Second Quarter Fiscal 2026 Financial Results Conference Call. All lines have been placed on mute to prevent any background noise. After the speakers' remarks, there will be a question and answer session. If you would like to ask a question during this time, simply press star followed by the number one on your telephone keypad. If you would like to withdraw your question, press star 1 again. Thank you. Toshiya Hari, you may begin your conference. Thank you.\n[0001] Operator (Operator): Good afternoon, everyone, and welcome to NVIDIA Corporation's conference call for 2026. With me today from NVIDIA Corporation are Jensen Huang, president and chief executive officer, and Colette Kress, executive vice president and chief financial officer. I would like to remind you th

In [20]:
def get_earnings_call_insights(client, call_transcripts: dict, model_name: str = "gpt-5-mini") -> EarningsCallInsights:
    """
    Obtiene insights clave de una transcripción de earnings call utilizando un modelo de OpenAI.

    Args:
        client: Cliente de OpenAI inicializado.
        transcript_text: El texto de la transcripción de la llamada.
        model_name: El nombre del modelo de OpenAI a utilizar.

    Returns:
        Un objeto diccionario derivado de EarningsCallInsights con los insights extraídos.
    """
    transcript_text = render_transcript(call_transcripts)
    response = client.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": "Eres un experto Analista Financiero. Devuelve SOLO un JSON válido que siga exactamente el esquema de EarningsCallInsights. Salidas en español."},
            {"role": "user", "content": transcript_text},
        ],
        response_format=EarningsCallInsights,
    )
    insights = response.choices[0].message.parsed
    return insights.model_dump()

In [21]:
%%time
insights = get_earnings_call_insights(client=client, call_transcripts=call_transcripts)

CPU times: user 26.1 ms, sys: 4.27 ms, total: 30.3 ms
Wall time: 33.5 s


In [22]:
insights.keys()

dict_keys(['symbol', 'sentiment', 'summary', 'key_topics', 'guidance', 'numeric_highlights', 'risks', 'catalysts', 'analyst_questions', 'unanswered_topics', 'bullish_points', 'bearish_points', 'red_flags', 'emotion'])

In [37]:
insights['summary']

'NVIDIA reportó un trimestre récord con ingresos totales de $46.7B y un crecimiento interanual del 56% en Data Center impulsado por la plataforma Blackwell y el inicio de envíos de GB300. La compañía guía conservadoramente Q3 sin asumir envíos H20 a China, pero espera $54B ±2% y márgenes brutos por encima del 73%. Riesgos geopolíticos (H20), un salto de inventario y mayor gasto operativo se contraponen a una sólida ejecución de producto, rampas de producción (≈1,000 racks/semana) y oportunidades de mercado masivas en infraestructura AI.'